# Reality AI — Production LoRA Fine-Tuning (Mistral-7B)

This notebook fine-tunes **Mistral-7B-v0.1** using QLoRA (4-bit quantised base
model + LoRA adapters) on the Reality AI property-chat dataset.

**Environment:** Kaggle GPU (T4 16 GB or P100 16 GB)  
**Dataset:** `full_dataset.jsonl` (~750 synthetic conversations)  
**Output:** LoRA adapter weights (zipped for download)  

The adapter teaches the model to behave as "RealityAI" — a Surat real-estate
chat assistant that understands property types, pricing in INR, Surat
neighbourhoods, and amenity-aware search.

### How this fits in the project
- **Person 1 (AI track)** trains and exports the adapter here.
- The adapter is downloaded and placed at `ai/finetune/adapters/reality-ai-lora/`.
- `ai/inference/` loads the adapter at runtime to power the chat endpoint.
- See `ai/docs/ADR-001-base-model.md` for the model selection rationale.

> **Kaggle GPU quota:** Free tier = 30 hrs/week. This run takes ~1-2 hrs on T4.
> Monitor wall-clock time in Cell 7 output.

> **IMPORTANT:** After running Cell 2 (pip install), you MUST restart the
> runtime (Runtime > Restart Session) before running Cell 3 onwards.

In [ ]:
# ============================================================================
# Cell 2: Install compatible dependencies
# ============================================================================
# Kaggle pre-installs torch and some HF libraries. We pin versions to
# avoid conflicts between bitsandbytes, transformers, and torch.
#
# AFTER RUNNING THIS CELL:
#   1. Click Runtime > Restart Session
#   2. Then run from Cell 3 onwards (skip this cell on re-run)

!pip install -q --upgrade \
    transformers==4.46.3 \
    peft==0.13.2 \
    bitsandbytes==0.45.0 \
    accelerate==1.1.1 \
    datasets>=2.20.0 \
    trl==0.12.2

print("\n" + "=" * 60)
print("  RESTART THE RUNTIME NOW")
print("  Runtime > Restart Session")
print("  Then run from Cell 3 onwards (skip this cell).")
print("=" * 60)

In [ ]:
# ============================================================================
# Cell 3: Imports and configuration constants
# ============================================================================

import os
import time
import shutil
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTTrainer

# trl 0.12.x uses TrainingArguments; trl 0.15+ uses SFTConfig.
# We pin trl==0.12.2 on Kaggle, so use TrainingArguments here.
from transformers import TrainingArguments

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_NAME: str = "mistralai/Mistral-7B-v0.1"

# Kaggle dataset path — adjust the dataset slug if yours is different.
DATASET_PATH: str = "/kaggle/input/reality-ai-finetune-data/full_dataset.jsonl"

OUTPUT_DIR: str = "/kaggle/working/reality-ai-lora"
ZIP_OUTPUT: str = "/kaggle/working/reality-ai-lora"  # .zip appended by shutil

# Training hyperparameters
NUM_EPOCHS: int = 3
LEARNING_RATE: float = 2e-4
PER_DEVICE_BATCH_SIZE: int = 2      # T4 16GB can handle batch=2
GRADIENT_ACCUMULATION_STEPS: int = 4  # effective batch = 8
MAX_SEQ_LENGTH: int = 2048

# ── Sanity checks ─────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU detected! Enable GPU in Kaggle settings."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_PATH}")
print(f"Effective batch size: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

In [ ]:
# ============================================================================
# Cell 4: Load and preprocess the dataset
# ============================================================================
# Mistral-7B-v0.1 is a BASE model — it has no built-in chat template.
# We define a standard ChatML template so the model learns a consistent
# turn-taking format during fine-tuning.

print(f"Loading dataset from: {DATASET_PATH}")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Dataset size: {len(dataset)} examples")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

# Set ChatML template — Mistral base model doesn't have one built in.
# This teaches the model the <|im_start|>/<|im_end|> turn structure.
CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{'<|im_start|>assistant\n'}}"
    "{% endif %}"
)
tokenizer.chat_template = CHATML_TEMPLATE

def apply_chat_template(example):
    """Format the messages list using the ChatML template."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template, remove_columns=dataset.column_names)
print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}")
print(f"\n... ({len(dataset[0]['text'])} chars total)")

In [ ]:
# ============================================================================
# Cell 5: Load the base model in 4-bit quantisation
# ============================================================================
# Even on a 16GB T4, 4-bit quantisation is worthwhile:
#   - Mistral-7B at fp16 = ~14 GB (barely fits, no room for training)
#   - Mistral-7B at 4-bit = ~5 GB (leaves ~11 GB for optimizer/activations)

# Detect compute dtype — T4 is Turing (fp16 only), P100 is Pascal (fp16 only)
if torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    print("Using bfloat16 compute dtype.")
else:
    compute_dtype = torch.float16
    print("Using float16 compute dtype (GPU does not support bf16).")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"\nLoading {MODEL_NAME} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("Model loaded.")

In [ ]:
# ============================================================================
# Cell 6: Apply LoRA adapters
# ============================================================================
# Values inlined from ai/finetune/configs/lora_config_full.yaml.
# If you change the YAML, update these to match.

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
pct = 100.0 * trainable / total
print(f"Trainable parameters: {trainable:,} / {total:,} ({pct:.2f}%)")

In [ ]:
# ============================================================================
# Cell 7: Configure and run training
# ============================================================================
# Using TrainingArguments (trl 0.12.x API).

use_bf16 = (compute_dtype == torch.bfloat16)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("=" * 60)
print(f"Starting training: {NUM_EPOCHS} epochs, "
      f"effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print("=" * 60)

t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0

mins, secs = divmod(int(elapsed), 60)
hrs, mins = divmod(mins, 60)
time_str = f"{hrs}h {mins}m {secs}s" if hrs else f"{mins}m {secs}s"

print("\n" + "=" * 60)
print(f"  Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Wall-clock:     {time_str}")
print(f"  Trainable:      {trainable:,} params ({pct:.2f}%)")
print(f"  GPU hours used: {elapsed / 3600:.2f} / 30.0 weekly quota")
print("=" * 60)

In [ ]:
# ============================================================================
# Cell 8: Save adapter and create downloadable zip
# ============================================================================

print(f"Saving adapter to {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# List saved files
saved_files = list(Path(OUTPUT_DIR).rglob("*"))
total_size_mb = sum(f.stat().st_size for f in saved_files if f.is_file()) / 1024**2
print(f"Adapter files: {len(saved_files)} files, {total_size_mb:.1f} MB total")
for f in sorted(saved_files):
    if f.is_file():
        print(f"  {f.relative_to(OUTPUT_DIR)}  ({f.stat().st_size / 1024:.0f} KB)")

# Create a single .zip for easy download from Kaggle's Output tab
zip_path = shutil.make_archive(ZIP_OUTPUT, "zip", root_dir=OUTPUT_DIR)
zip_size_mb = Path(zip_path).stat().st_size / 1024**2
print(f"\nZipped adapter: {zip_path} ({zip_size_mb:.1f} MB)")
print("Download this from the Output tab on the right. -->")

## Next Steps

### 1. Download the adapter
- Click the **Output** tab in the right sidebar of this Kaggle notebook.
- Download `reality-ai-lora.zip`.

### 2. Place it in your local repo
```
ai/finetune/adapters/reality-ai-lora/
    adapter_config.json
    adapter_model.safetensors
    tokenizer_config.json
    tokenizer.json
    special_tokens_map.json
    ...
```
Unzip the downloaded file into `ai/finetune/adapters/reality-ai-lora/`.

### 3. Load it in inference
The `ai/inference/` module will load this adapter at runtime:
```python
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained(
    "ai/finetune/adapters/reality-ai-lora/",
    device_map="auto",
    torch_dtype=torch.float16,
)
```

### 4. Don't commit the adapter to git
The `ai/finetune/adapters/` directory is already in `.gitignore`.
Store the zip in Google Drive or Kaggle Datasets for team access.

> **Re-training:** If you update the dataset, just re-run this notebook.
> The full run takes ~1-2 hours on a T4 GPU.